# Семинар 5.2: Дополнительные задания (очень простой вариант)

Минимальный ноутбук без классов и без сложной структуры.


In [14]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42


In [15]:
def path_of(name):
    for p in [Path(name), Path('5.2') / name]:
        if p.exists():
            return p
    raise FileNotFoundError(name)

prsa_path = path_of('PRSA_Data.csv')
titanic_path = path_of('titanic.csv')

prsa = pd.read_csv(prsa_path, index_col=0)
print('PRSA:', prsa.shape)
prsa.head(2)


PRSA: (35064, 10)


,No,SO2,NO2,CO,O3,PRES,RAIN,wd,WSPM,AQI Label
0,1,6.0,28.0,400.0,51.577659,1023.0,0.0,NNW,4.4,Severely Polluted
1,2,6.0,28.0,400.0,50.403851,1023.2,0.0,N,4.7,Severely Polluted


## 1) PRSA: короткий pipeline для всех экспериментов

In [16]:
WIND_TO_DEGREES = {
    'N':0,'NNE':22.5,'NE':45,'ENE':67.5,'E':90,'ESE':112.5,'SE':135,'SSE':157.5,
    'S':180,'SSW':202.5,'SW':225,'WSW':247.5,'W':270,'WNW':292.5,'NW':315,'NNW':337.5
}


def fit_params(train_df, cfg):
    train_df = train_df.replace(-1, np.nan).copy()
    p = {'clip': {}, 'co_bins': None}

    if cfg.get('clip_q'):
        ql, qh = cfg['clip_q']
        for c in train_df.select_dtypes(include=np.number).columns:
            s = train_df[c].dropna()
            if len(s):
                p['clip'][c] = (float(s.quantile(ql)), float(s.quantile(qh)))

    if cfg.get('co_bin') and 'CO' in train_df.columns:
        s = train_df['CO'].dropna()
        if len(s):
            q33, q66 = s.quantile([0.33, 0.66])
            lo, hi = float(s.min()) - 1e-9, float(s.max()) + 1e-9
            bins = np.unique(np.array([lo, q33, q66, hi]))
            if len(bins) < 4:
                bins = np.linspace(lo, hi, 4)
            p['co_bins'] = bins

    return p


def apply_features(df, cfg, p):
    x = df.replace(-1, np.nan).copy()

    for c, (lo, hi) in p.get('clip', {}).items():
        if c in x.columns:
            x[c] = x[c].clip(lo, hi)

    if cfg.get('rain_bin') and 'RAIN' in x.columns:
        x['IS_RAIN'] = (x['RAIN'] > 0).astype(float)
        x = x.drop(columns=['RAIN'])

    if cfg.get('co_bin') and p.get('co_bins') is not None and 'CO' in x.columns:
        x['CO_BIN'] = pd.cut(x['CO'], bins=p['co_bins'], labels=False, include_lowest=True).astype(float)

    if cfg.get('round_o3') and 'O3' in x.columns:
        x['O3'] = np.round(x['O3'], 0)

    if cfg.get('log_so2') and 'SO2' in x.columns:
        x['SO2'] = np.where(x['SO2'] >= 0, np.log1p(x['SO2']), np.nan)

    if cfg.get('ratios'):
        if {'NO2','SO2'}.issubset(x.columns):
            x['NO2_SO2_RATIO'] = x['NO2'] / (x['SO2'].abs() + 1)
        if {'CO','O3'}.issubset(x.columns):
            x['CO_O3_RATIO'] = x['CO'] / (x['O3'].abs() + 1)

    if cfg.get('wind_sincos') and 'wd' in x.columns:
        deg = x['wd'].map(WIND_TO_DEGREES)
        rad = np.deg2rad(deg)
        x['wd_sin'] = np.sin(rad)
        x['wd_cos'] = np.cos(rad)

    return x


def make_pipe(imputer='median', use_knn=False):
    num_pipe = Pipeline([
        ('imputer', KNNImputer(n_neighbors=5) if use_knn else SimpleImputer(strategy=imputer)),
        ('scaler', RobustScaler()),
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('oh', OneHotEncoder(handle_unknown='ignore')),
    ])
    prep = ColumnTransformer([
        ('num', num_pipe, make_column_selector(dtype_include=np.number)),
        ('cat', cat_pipe, make_column_selector(dtype_exclude=np.number)),
    ])
    model = LogisticRegression(max_iter=3000, solver='saga', random_state=RANDOM_STATE, n_jobs=-1)
    return Pipeline([('prep', prep), ('model', model)])


def run_exp(X_train, y_train, X_test, y_test, cfg, imputer='median', use_knn=False, cv=3):
    p = fit_params(X_train, cfg)
    Xtr = apply_features(X_train, cfg, p)
    Xte = apply_features(X_test, cfg, p)
    pipe = make_pipe(imputer=imputer, use_knn=use_knn)

    cv_obj = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(pipe, Xtr, y_train, cv=cv_obj, scoring='accuracy', n_jobs=-1)

    pipe.fit(Xtr, y_train)
    pred = pipe.predict(Xte)
    return {
        'cv_acc': float(cv_scores.mean()),
        'test_acc': float(accuracy_score(y_test, pred)),
        'test_f1_macro': float(f1_score(y_test, pred, average='macro')),
        'pipe': pipe,
        'params': p,
        'pred': pred,
    }


In [17]:
X = prsa.drop(columns=['AQI Label'])
y = prsa['AQI Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print('train/test:', X_train.shape, X_test.shape)


train/test: (28051, 9) (7013, 9)


## 2) Доп. задание 1: эффект необязательных преобразований

In [ ]:
cfgs_optional = {
    'baseline': {},
    'RAIN->IS_RAIN': {'rain_bin': True},
    'CO binning': {'co_bin': True},
    'adaptive clipping': {'clip_q': (0.01, 0.99)},
    'O3 rounding': {'round_o3': True},
    'SO2 log1p': {'log_so2': True},
}

rows=[]
for name, cfg in cfgs_optional.items():
    m = run_exp(X_train, y_train, X_test, y_test, cfg)
    rows.append({'experiment': name, 'cv_acc': m['cv_acc'], 'test_acc': m['test_acc'], 'test_f1_macro': m['test_f1_macro']})

optional_results = pd.DataFrame(rows).sort_values('test_acc', ascending=False)
optional_results


In [ ]:
sns.barplot(data=optional_results, y='experiment', x='test_acc', orient='h', color='#4e79a7')
plt.title('PRSA: необязательные преобразования')
plt.xlabel('Test accuracy')
plt.ylabel('')
plt.tight_layout()
plt.show()


## 3) Доп. задание 2: методы заполнения пропусков

In [ ]:
base_cfg = {'rain_bin': True, 'co_bin': True, 'clip_q': (0.01, 0.99), 'round_o3': True, 'log_so2': True}

imputer_cfgs = {
    'median': ('median', False),
    'mean': ('mean', False),
    'most_frequent': ('most_frequent', False),
    'knn': ('median', True),
}

rows=[]
for name, (imp, knn) in imputer_cfgs.items():
    m = run_exp(X_train, y_train, X_test, y_test, base_cfg, imputer=imp, use_knn=knn)
    rows.append({'experiment': name, 'cv_acc': m['cv_acc'], 'test_acc': m['test_acc'], 'test_f1_macro': m['test_f1_macro']})

imputation_results = pd.DataFrame(rows).sort_values('test_acc', ascending=False)
imputation_results


## 4) Доп. задание 3: дополнительные фичи

In [ ]:
feature_cfgs = {
    'base': base_cfg,
    'base + ratios': {**base_cfg, 'ratios': True},
    'base + wind sin/cos': {**base_cfg, 'wind_sincos': True},
    'base + ratios + wind': {**base_cfg, 'ratios': True, 'wind_sincos': True},
}

rows=[]
for name, cfg in feature_cfgs.items():
    m = run_exp(X_train, y_train, X_test, y_test, cfg)
    rows.append({'experiment': name, 'cv_acc': m['cv_acc'], 'test_acc': m['test_acc'], 'test_f1_macro': m['test_f1_macro']})

feature_results = pd.DataFrame(rows).sort_values('test_acc', ascending=False)
feature_results


## 5) Доп. задания 4-5: train/test до очистки и воспроизводимость

In [ ]:
best_feature_name = feature_results.iloc[0]['experiment']
best_imputer_name = imputation_results.iloc[0]['experiment']

best_cfg = feature_cfgs[best_feature_name]
best_imp, best_knn = imputer_cfgs[best_imputer_name]

final_run = run_exp(X_train, y_train, X_test, y_test, best_cfg, imputer=best_imp, use_knn=best_knn, cv=5)
print('best feature setup:', best_feature_name)
print('best imputer setup:', best_imputer_name)
print('metrics:', {'cv_acc': final_run['cv_acc'], 'test_acc': final_run['test_acc'], 'test_f1_macro': final_run['test_f1_macro']})
print(classification_report(y_test, final_run['pred']))

artifact = {
    'feature_cfg': best_cfg,
    'feature_params': final_run['params'],
    'model_pipe': final_run['pipe'],
}
Path('artifacts').mkdir(exist_ok=True)
joblib.dump(artifact, 'artifacts/prsa_simple_artifact.joblib')
print('saved: artifacts/prsa_simple_artifact.joblib')


## 6) Доп. задание 6: даты в Customer_support

In [ ]:
def add_time_parts(df, date_cols):
    out = df.copy()
    for c in date_cols:
        dt = pd.to_datetime(out[c], errors='coerce', utc=True)
        out[c + '_dayofweek'] = dt.dt.dayofweek
        out[c + '_hour'] = dt.dt.hour
        out[c + '_day'] = dt.dt.day
        out[c + '_part_of_day'] = pd.cut(dt.dt.hour, bins=[-1,5,11,17,23], labels=['night','morning','day','evening'])
    return out

candidates = [Path('Customer_support.csv'), Path('customer_support.csv'), Path('5.1/Customer_support.csv'), Path('5.1/customer_support.csv')]
cs_path = next((p for p in candidates if p.exists()), None)

if cs_path is None:
    print('Customer_support.csv не найден.')
else:
    cs = pd.read_csv(cs_path)
    date_cols = []
    for c in cs.columns:
        if cs[c].dtype == 'object':
            ok_ratio = pd.to_datetime(cs[c], errors='coerce', utc=True).notna().mean()
            if ok_ratio > 0.7:
                date_cols.append(c)
    cs2 = add_time_parts(cs, date_cols)
    print('date columns:', date_cols)
    cs2.head(2)


## 7) Доп. задание 7: Titanic (коротко)

In [ ]:
tit = pd.read_csv(titanic_path)

X = tit.drop(columns=['Survived']).copy()
y = tit['Survived'].astype(int)

# простые фичи
X['Title'] = X['Name'].str.extract(r',\s*([^\.]+)\.', expand=False)
X['FamilySize'] = X['SibSp'] + X['Parch'] + 1
X['IsAlone'] = (X['FamilySize'] == 1).astype(float)
X['CabinKnown'] = X['Cabin'].notna().astype(float)
X = X.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())])
cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))])
prep = ColumnTransformer([
    ('num', num_pipe, make_column_selector(dtype_include=np.number)),
    ('cat', cat_pipe, make_column_selector(dtype_exclude=np.number)),
])
pipe = Pipeline([('prep', prep), ('model', LogisticRegression(max_iter=3000, solver='liblinear', random_state=RANDOM_STATE))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

print({'cv_acc': float(cv_scores.mean()), 'test_acc': float(accuracy_score(y_test, pred)), 'test_f1_macro': float(f1_score(y_test, pred, average='macro'))})
print(classification_report(y_test, pred))


## 8) Доп. задание 8 (*): автоочистка любого датасета без классов

In [ ]:
def auto_clean_numeric(df, missing_tokens=(-1, -999, -9999), clip_q=(0.01, 0.99)):
    x = df.copy()
    for t in missing_tokens:
        x = x.replace(t, np.nan)

    num_cols = x.select_dtypes(include=np.number).columns.tolist()
    report = []

    for c in num_cols:
        s = pd.to_numeric(x[c], errors='coerce')
        lo, hi = s.quantile(clip_q[0]), s.quantile(clip_q[1])
        x[c] = s.clip(lo, hi)
        if s.min(skipna=True) >= 0 and abs(s.skew(skipna=True)) > 1:
            x[c] = np.log1p(x[c])
            used_log = True
        else:
            used_log = False
        x[c] = x[c].fillna(x[c].median())
        report.append({'column': c, 'clip_low': float(lo), 'clip_high': float(hi), 'log1p': used_log})

    return x, pd.DataFrame(report)

prsa_auto, prsa_auto_report = auto_clean_numeric(prsa.drop(columns=['AQI Label']), missing_tokens=(-1,))
print('auto cleaned shape:', prsa_auto.shape)
prsa_auto_report.head()
